# The Model Registry: versions, aliases, and the promotion lifecycle

This is the third **advanced** tutorial in the repo. It assumes you've worked through:

- `c_hyperparameter_tuning.ipynb` — which ended with a *one-line* `mlflow.register_model(...)` and a one-sentence mention of a `champion` alias.
- `f_model_evaluation.ipynb` — which built a **validation gate** (`validate_evaluation_results`) and then said, in its last cell: *"once a candidate passes its gate, registering it plus an alias is what actually moves it into the serving path. A future notebook will cover the registry + alias workflow end-to-end."*

This is that notebook.

So far the registry has been a footnote: you *registered* a model but never managed it. This notebook covers the part that turns the registry from a glorified folder into the backbone of an ML deployment process:

1. **Versions** — why registering never overwrites.
2. **Aliases** — the mutable named pointer (`@champion`) that production code and `mlflow models serve` actually reference.
3. **The promotion gate** — register a challenger, compare it to the champion, and promote it by *moving one alias*.
4. **Rollback + governance** — undo a promotion with a single call, and annotate versions with tags and descriptions so the registry records *why* each version exists.

## The problem: an opaque model URI is not a deployment target

Every notebook so far loaded models from a hash-based URI, e.g.:

```python
mlflow.pyfunc.load_model("runs:/3f9a1c8e7b2d4.../model")
```

That URI is an opaque hash tied to one specific logged model. It works in *your* notebook, the day you trained the model. It falls apart the moment anyone else is involved:

- **A teammate** asks "which run is the one we actually shipped?" — and the answer lives in your head, a Slack thread, or a spreadsheet cell.
- **A serving process** (next notebook) needs a *stable* address to load from. It cannot hardcode a run id that changes every time you retrain.
- **A rollback** ("revert to last week's model") means digging through the runs table for the right hash.

The **Model Registry** is the layer that fixes this. It sits on top of runs and adds:

- a **name** you choose (`ca_housing_price`), not a hash,
- **versions** that accumulate as you retrain, and
- **aliases** — movable labels like `@champion` that always point at *whatever version is current*, so the rest of your system never has to know a version number.

The concern this serves is **governance + deployment**: decoupling *"which artifact is in production"* from *"which run produced it"*.

## Four registry concepts

Define these once; the rest of the notebook uses them constantly.

| Term | What it is | Example |
|---|---|---|
| **Registered model** | A named container in the registry. Has no weights itself — it holds versions. | `ca_housing_price` |
| **Model version** | One registered artifact. Auto-incremented integer. Immutable once created. | `ca_housing_price` **v3** |
| **Alias** | A *mutable, named pointer* to one version. Move it freely; it's how you say "this is the live one". | `@champion` → v3 |
| **Tag** | A key–value label on a model or version. Metadata, not a pointer. | `validation_status=approved` |

The mental model: **versions are immutable history; aliases are mutable bookmarks into that history; tags are sticky notes.**

## Version drift: model **Stages** are gone — use **aliases** + **tags**

**Stale in upstream tutorial:** a large fraction of MLflow blog posts, Stack Overflow answers, and pre-3.x tutorials manage the lifecycle with **Stages** — a fixed enum of `None` / `Staging` / `Production` / `Archived` — moved with:

```python
# Deprecated since MLflow 2.9. Do not use in new code.
client.transition_model_version_stage(
    name="ca_housing_price", version=3, stage="Production",
)
```

Stages were **deprecated in MLflow 2.9** and replaced by the more flexible **aliases + tags** pair (introduced in 2.8). In MLflow 3 the old stage API still exists only as a thin backward-compat shim. If you follow an older tutorial that calls `transition_model_version_stage` you'll either hit a deprecation warning or fight an abstraction the rest of MLflow 3 has moved away from.

Why the change? Stages were a *fixed* enum — exactly four labels, one model per stage. Real teams need more: a `@champion` and three regional `@champion-eu` / `@champion-us` pointers, a `@challenger` under test, a `@shadow` model logging predictions but not serving. Aliases are arbitrary names you invent; tags carry the status metadata that the four stages used to encode.

| Old (Stages, pre-3.x) | New (MLflow 3) |
|---|---|
| `stage="Production"` | alias `@champion` (or `@production`) |
| `stage="Staging"` | alias `@challenger` (or `@staging`) |
| `stage="Archived"` | just remove the alias; the version stays as history |
| `stage="None"` | a version with no alias pointing at it |
| (no equivalent) | tag `validation_status=approved`, `owner=...`, any key you like |

This notebook uses `@champion` / `@challenger`. Those names are conventions, not keywords — `@prod` would work identically.

## Prerequisites

### Start the tracking server

**Missing from upstream tutorial:** before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal** from the repo root:

```bash
mlflow ui --host 127.0.0.1 --port 5001
```

(The advanced notebooks in this repo — `e_`, `f_`, and this one — all use port `5001`.)

No new dependencies: this notebook uses only `scikit-learn` and `mlflow`, both already in `pyproject.toml`.

In [1]:
import mlflow
from mlflow import MlflowClient
from mlflow.exceptions import RestException
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

HOST, PORT = "127.0.0.1", 5001
mlflow.set_tracking_uri(f"http://{HOST}:{PORT}")

# The client is the registry API surface — register_model is a thin
# wrapper, but aliases, tags, and search only exist on MlflowClient.
client = MlflowClient()

# Idempotent experiment creation (principle 7): re-running is a no-op.
EXPERIMENT = "Model Registry"
try:
    mlflow.create_experiment(EXPERIMENT)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(EXPERIMENT)

# One canonical registered-model name for the whole notebook.
MODEL_NAME = "ca_housing_price"

## Step 1: Train and log a first model

Nothing new here — the same California-housing `RandomForestRegressor` from `f_`, logged inside a run with a signature so downstream consumers know its input/output schema. We hold out a test split so we can score versions against each other in Step 5.

We capture `champion_info`, whose `.model_uri` is the opaque, auto-generated address of the just-logged model — exactly the kind of URI we're about to wrap in a stable registered name.

**What changed in MLflow 3:** earlier notebooks (and most older tutorials) refer to the logged model as `runs:/<run_id>/model`. MLflow 3 introduced first-class **logged models**, each with its own id, so `log_model(...).model_uri` now comes back as `models:/m-<hash>` instead. Both forms are opaque hashes, both still work as a registration source, and the registry's whole job is to put a memorable name in front of either one — so the change doesn't affect anything below.

In [2]:
raw = fetch_california_housing(as_frame=True)
X, y = raw.data, raw.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

champion_model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0)
champion_model.fit(X_train, y_train)
signature = infer_signature(X_train, champion_model.predict(X_train))

with mlflow.start_run(run_name="champion_candidate"):
    champion_info = mlflow.sklearn.log_model(
        champion_model, name="model",
        signature=signature, input_example=X_train.head(),
    )

print("logged model URI (opaque, auto-generated):", champion_info.model_uri)

2026/05/29 14:18:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/05/29 14:18:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run champion_candidate at: http://127.0.0.1:5001/#/experiments/1/runs/8515774b1f5143d186a2c685dffa8275
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
logged model URI (opaque, auto-generated): models:/m-f4b5f8c2fc00495e9d5d9331fd90bbf8


## Step 2: Register version 1

`mlflow.register_model(model_uri, name)` copies the run's logged model into the registry under `name`, returning a `ModelVersion`. The first call creates **version 1**.

**Registering never overwrites.** Run this cell again (or retrain and register again) and you get version 2, 3, 4 … — the registry is append-only, so every model you ever promoted stays reachable for rollback and audit.

Because of that, we don't hardcode the integer `1`. We stash the version this run produced in `champion_version` and use that variable everywhere downstream — so the notebook stays correct on a second **Run All** (where it'll be version 2, 3, …) instead of silently pointing at a stale version.

In [3]:
registered = mlflow.register_model(model_uri=champion_info.model_uri, name=MODEL_NAME)
champion_version = registered.version

print(f"Registered '{registered.name}' as version {champion_version}")

Successfully registered model 'ca_housing_price'.
2026/05/29 14:18:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ca_housing_price, version 1


Created version '1' of model 'ca_housing_price'.


Registered 'ca_housing_price' as version 1


## Step 3: Aliases — the name production code actually points at

A version number is *better* than a run hash, but it's still brittle: every consumer that hardcodes `models:/ca_housing_price/1` has to be found and edited the day you ship version 2. That's the same coupling problem one level up.

An **alias** breaks it. `set_registered_model_alias(name, alias, version)` pins a mutable label to a version. Production code, dashboards, and the serving process all reference the *alias*; when you promote a new version you move the alias and **nothing downstream changes**.

Compare the three ways to address the same artifact:

| URI | Stability | Who uses it |
|---|---|---|
| `runs:/<hash>/model` | tied to one run, opaque | the training notebook, briefly |
| `models:/ca_housing_price/3` | stable but version-pinned | reproducing a specific past result |
| `models:/ca_housing_price@champion` | **follows promotions automatically** | production, serving, dashboards |

Setting an alias is idempotent — re-running just re-points it at the same version, so this cell is safe to run twice.

In [4]:
client.set_registered_model_alias(
    name=MODEL_NAME, alias="champion", version=champion_version,
)

# Production code loads by alias and never has to know the version number.
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
preds = champion.predict(X_test.head())

print(f"@champion currently resolves to version {champion_version}")
print("sample predictions:", preds.round(3).tolist())

@champion currently resolves to version 1
sample predictions: [1.729, 2.73, 1.699, 1.22, 4.205]


## Step 4: A challenger — register a second candidate

Retraining is the normal case: new data arrives, you try richer hyperparameters, you produce a *candidate* that might be better. You do **not** want to overwrite the live model on faith — you want to stage the candidate alongside it and decide on evidence.

We train a deeper forest, register it (this becomes the next version), and give it the `@challenger` *alias* — an alias, not a tag: it's a movable pointer to a version, exactly like `@champion`. Now the registry holds two live pointers: `@champion` (serving) and `@challenger` (under evaluation).

In [5]:
challenger_model = RandomForestRegressor(n_estimators=300, max_depth=16, random_state=0)
challenger_model.fit(X_train, y_train)

with mlflow.start_run(run_name="challenger_candidate"):
    challenger_info = mlflow.sklearn.log_model(
        challenger_model, name="model",
        signature=signature, input_example=X_train.head(),
    )

challenger_version = mlflow.register_model(
    model_uri=challenger_info.model_uri, name=MODEL_NAME,
).version
client.set_registered_model_alias(
    name=MODEL_NAME, alias="challenger", version=challenger_version,
)

print(f"@champion   -> version {champion_version}")
print(f"@challenger -> version {challenger_version}")

2026/05/29 14:18:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/05/29 14:18:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run challenger_candidate at: http://127.0.0.1:5001/#/experiments/1/runs/4d8c7cb8acb54228a4bf0a4468de9393
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


Registered model 'ca_housing_price' already exists. Creating a new version of this model...
2026/05/29 14:18:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ca_housing_price, version 2


Created version '2' of model 'ca_housing_price'.


@champion   -> version 1
@challenger -> version 2


## Step 5: The promotion gate — only promote if the challenger wins

This is the whole point of the two-pointer setup. Promotion is a **decision**, and the decision should be evidence-based, not "the new one is probably better."

`f_model_evaluation.ipynb` built the formal CI version of this gate: `mlflow.validate_evaluation_results(candidate_result, baseline_result, thresholds)`, which raises if the candidate doesn't beat the baseline by a margin. That's the right tool in a CI pipeline, and the place it belongs is *right here*, between registration and promotion — re-read `f_` Step 7 for the full `MetricThreshold` API.

To keep this notebook about the **registry** rather than re-teaching evaluation, we use a deliberately simple gate: score both versions on the held-out set and promote only if the challenger's RMSE is lower. The mechanics of promotion are what matter — and they're tiny: **promotion is one `set_registered_model_alias` call** that moves `@champion` onto the winning version.

Notice what we *don't* do: touch the serving process, edit any consumer, or delete the old version. The old champion stays in the registry as version history, one alias-move away from being restored.

In [6]:
def rmse_of(alias_or_version_uri):
    model = mlflow.pyfunc.load_model(alias_or_version_uri)
    return root_mean_squared_error(y_test, model.predict(X_test))

champion_rmse = rmse_of(f"models:/{MODEL_NAME}/{champion_version}")
challenger_rmse = rmse_of(f"models:/{MODEL_NAME}/{challenger_version}")

print(f"champion  (v{champion_version}) RMSE: {champion_rmse:.4f}")
print(f"challenger(v{challenger_version}) RMSE: {challenger_rmse:.4f}")

if challenger_rmse < champion_rmse:
    # The promotion: move @champion onto the challenger. One call.
    client.set_registered_model_alias(MODEL_NAME, "champion", challenger_version)
    promoted_version = challenger_version
    print(f"\nPROMOTED: @champion now -> version {challenger_version}")
else:
    promoted_version = champion_version
    print(f"\nKEPT: challenger did not beat champion; @champion stays at v{champion_version}")

# Confirm by resolving the alias from scratch.
live = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"resolved @champion -> version {live.version}")

champion  (v1) RMSE: 0.5963
challenger(v2) RMSE: 0.5233

PROMOTED: @champion now -> version 2
resolved @champion -> version 2


## Step 6: Rollback and governance

**Rollback is the payoff of the decoupling.** If the new champion misbehaves in production, you don't retrain or redeploy — you move the alias back. One call, instant, and the previous version was never deleted:

```python
client.set_registered_model_alias(MODEL_NAME, "champion", champion_version)  # undo
```

We won't leave it rolled back (we want the better model live), but the cell below shows the move both ways so you can see how cheap it is.

**Governance: tags and descriptions** record *why* a version exists and what the model *is* — things a version number alone never tells you. Tags come at **two levels** (recall the concept table: a tag lives "on a model **or** version"):

- **`set_model_version_tag(name, version, k, v)`** — status of *one version*: `validation_status=approved`. This is the machine-readable label the old Stages enum used to carry, now an arbitrary key–value you control.
- **`set_registered_model_tag(name, k, v)`** — facts about the *whole model*, true across every version: `task=regression`, `owner=pricing-team`. A version tag answers "is *this one* approved?"; a model tag answers "what *is* this model, and whose is it?".
- **`update_model_version(description=...)`** — a free-text human note on a version.
- **`delete_registered_model_alias`** — once the challenger *is* the champion, the `@challenger` pointer is stale; remove it. The version stays; only the label goes.

In [7]:
# --- rollback is one call (shown, then immediately re-applied) -------------
client.set_registered_model_alias(MODEL_NAME, "champion", champion_version)
print(f"rolled back: @champion -> v{client.get_model_version_by_alias(MODEL_NAME, 'champion').version}")
client.set_registered_model_alias(MODEL_NAME, "champion", promoted_version)
print(f"re-promoted: @champion -> v{client.get_model_version_by_alias(MODEL_NAME, 'champion').version}")

# --- governance metadata --------------------------------------------------
# Version-level tag: the status of THIS version (replaces the old Stages enum).
client.set_model_version_tag(MODEL_NAME, promoted_version, "validation_status", "approved")
# Registered-model-level tags: facts about the model as a whole, not any one version.
client.set_registered_model_tag(MODEL_NAME, "task", "regression")
client.set_registered_model_tag(MODEL_NAME, "owner", "pricing-team")
# Free-text description on the live version.
client.update_model_version(
    MODEL_NAME, promoted_version,
    description="Promoted by g_model_registry: lower held-out RMSE than prior champion.",
)

# --- retire the @challenger label now that it has been promoted -----------
if promoted_version == challenger_version:
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
    print("retired @challenger alias (its version lives on as history)")

rolled back: @champion -> v1
re-promoted: @champion -> v2


retired @challenger alias (its version lives on as history)


## Step 7: Inspect the registry

Two ways to see what you built — programmatically (for CI) and in the UI (for humans).

`search_model_versions` is the queryable view of everything under a name: every version with its tags. This is what a deployment script calls to answer *"what's live right now?"* without a human in the loop.

Tags split by level, mirroring how we set them: **version tags** live on each `ModelVersion` (`mv.tags`), while **model-level tags** live on the registered model (`client.get_registered_model(name).tags`). The cell below prints both.

**Gotcha — aliases don't live on the version objects `search` returns.** The `ModelVersion` rows from `search_model_versions` come back with an empty `.aliases` list. The authoritative alias→version mapping is a single dict on the *registered model*: `client.get_registered_model(name).aliases`. We fetch that once and invert it, rather than trusting `mv.aliases` per row.

In [8]:
# Alias map and model-level tags both live on the registered model.
rm = client.get_registered_model(MODEL_NAME)
print("model-level tags:", dict(rm.tags))

aliases_of = {}  # invert {alias: version} -> {version: [aliases]}
for alias, ver in rm.aliases.items():
    aliases_of.setdefault(ver, []).append(alias)

print("\nversions:")
for mv in sorted(client.search_model_versions(f"name='{MODEL_NAME}'"),
                 key=lambda m: int(m.version)):
    aliases = ", ".join(aliases_of.get(mv.version, [])) or "-"
    status = mv.tags.get("validation_status", "-")  # a version-level tag
    print(f"  v{mv.version:<3} aliases=[{aliases:<12}] validation_status={status}")

model-level tags: {'task': 'regression', 'owner': 'pricing-team'}

versions:
  v1   aliases=[-           ] validation_status=-
  v2   aliases=[champion    ] validation_status=approved


**In the UI:** open <http://127.0.0.1:5001/> and click the **Models** tab (top nav), then `ca_housing_price`.

- Each **version** row shows its aliases as colored chips and its tags.
- The `@champion` chip is on the promoted version; there is no `@challenger` chip anymore (Step 6 retired it).
- Click a version to read the **description** you set and the link back to the **source run** — the lineage from "live model" all the way to the run, parameters, and metrics that produced it.

That lineage — alias → version → run → params/metrics/artifacts — is the reproducibility story the whole repo has been building toward: a model in production you can trace back to exactly how it was made.

## Next steps

The alias you just created is not just documentation — it's an **address**. The next notebook serves it:

```bash
mlflow models serve -m "models:/ca_housing_price@champion" --port 5002
```

`mlflow models serve` stands up a REST endpoint in front of `@champion`. When you promote a new version (move the alias), the served model follows — *without restarting the consumers*. That's the link this notebook was built to enable.

- **[`h_model_serving.ipynb`](./h_model_serving.ipynb)** — serve the registered model over REST, the `/invocations` contract, and the Docker path. *(Next in this repo.)*
- **[Model Registry docs](https://mlflow.org/docs/latest/ml/model-registry/)** — the full concept guide.
- **[Registry workflows](https://mlflow.org/docs/latest/ml/model-registry/workflow/)** — `MlflowClient` recipes for aliases, tags, and search.
- **Webhooks / CI** — in a real org, promotion is triggered by a passing CI job (the `f_` gate), and a registry **webhook** can notify the serving system to reload. That's the automated version of the manual alias-move you did in Step 5.